In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

from auditorai.adapters.base import wrap
from auditorai.core.system import AuditorSystem

# Load dataset and split
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Train primary and wrap
primary_clf = RandomForestClassifier(n_estimators=100, random_state=42)
primary_clf.fit(X_train, y_train)
primary_adapter = wrap(primary_clf)

# Initialize and train AuditorSystem
system = AuditorSystem(primary_adapter)
system.train(X_val, y_val)

# Run auto_tune
best_tau = system.auto_tune(X_val, y_val)
print(f"Best auto-tuned threshold: {best_tau:.4f}")

# Get auditor predictions (p_wrong) for ROC curve
p_wrong_test = system.auditor_.p_wrong(X_test, primary_adapter)
# True errors made by primary model
y_pred_test = primary_adapter.predict(X_test)
errors_test = (y_pred_test != y_test).astype(int)

# Plot ROC curve
fpr, tpr, _ = roc_curve(errors_test, p_wrong_test)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Auditor ROC Curve")
plt.legend(loc="lower right")

# Plot calibration curve
prob_true, prob_pred = calibration_curve(errors_test, p_wrong_test, n_bins=5)
plt.subplot(1, 2, 2)
plt.plot(prob_pred, prob_true, "s-", label="Auditor")
plt.plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
plt.xlabel("Mean Predicted Probability of Error")
plt.ylabel("Fraction of Actual Errors")
plt.title("Auditor Calibration Curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()